# 03 – Data Pipeline

**Goal:** Run the end-to-end data pipeline — from raw curated data to model-ready train / val / test splits stored in S3.

**Input:** `s3://ads508-team1-sephora/curated/`  
**Output:** `s3://ads508-team1-sephora/Model/final_splits/`

---

| Section | Description |
|---|---|
| 1 | **Import Libraries** — all dependencies |
| 2 | **Pipeline Function** — `run_product_pipeline()` applies all transformation & feature engineering steps in one call |
| 3 | **Save Final Splits** — train / val / test splits uploaded to S3 |

## Import Libraries

In [9]:
import boto3
import pandas as pd
import numpy as np
import sklearn
import sagemaker
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
import re
import nltk
import time
from botocore.exceptions import BotoCoreError, ClientError
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import os
import shutil
import warnings
warnings.filterwarnings("ignore")

## Pipeline

In [10]:
# Define the S3 path
s3_path = 's3://ads508-team1-sephora/curated/products/products.parquet'

# Load the parquet directly into a DataFrame
df_raw = pd.read_parquet(s3_path)


def run_product_pipeline(df_raw, top_n_secondary=5, n_svd_components=10):
    # 1. INITIAL FILTERING
    df = (
        df_raw[df_raw['primary_category'] == 'Makeup']
        .copy()
        .reset_index(drop=True)
    )

    # 2. IMPUTATION
    df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
    df['reviews'] = pd.to_numeric(df['reviews'], errors='coerce').fillna(0)
    df['loves_count'] = pd.to_numeric(df['loves_count'], errors='coerce').fillna(0)

    cols_to_fix = [
        'size', 'variation_type', 'variation_value', 'variation_desc',
        'ingredients', 'highlights', 'secondary_category', 'tertiary_category'
    ]
    for col in cols_to_fix:
        df[col] = df[col].fillna("NA")

    df['price_usd'] = pd.to_numeric(df['price_usd'], errors='coerce')
    df['value_price_usd'] = pd.to_numeric(df['value_price_usd'], errors='coerce')
    df['sale_price_usd'] = pd.to_numeric(df['sale_price_usd'], errors='coerce')
    df['child_max_price'] = pd.to_numeric(df['child_max_price'], errors='coerce')
    df['child_min_price'] = pd.to_numeric(df['child_min_price'], errors='coerce')

    df['price_usd'] = df['price_usd'].fillna(df['price_usd'].median())
    df['value_price_usd'] = df['value_price_usd'].fillna(df['price_usd'])
    df['sale_price_usd'] = df['sale_price_usd'].fillna(df['price_usd'])
    df['child_max_price'] = df['child_max_price'].fillna(0)
    df['child_min_price'] = df['child_min_price'].fillna(0)

    bool_cols = ['limited_edition', 'new', 'online_only', 'out_of_stock', 'sephora_exclusive']
    for col in bool_cols:
        if col in df.columns:
            df[col] = df[col].fillna(False).astype(int)
        else:
            df[col] = 0

    # 3. CORE NUMERICAL FEATURES
    df['price_log'] = np.log1p(df['price_usd'].clip(lower=0))
    df['log_loves'] = np.log1p(df['loves_count'].clip(lower=0))
    df['reviews_log'] = np.log1p(df['reviews'].clip(lower=0))

    df['num_ingredients'] = (
        df['ingredients']
        .fillna('')
        .apply(lambda x: len([i for i in str(x).split(',') if str(i).strip() not in ['', 'NA']]))
    )

    df['price_x_num_ing'] = df['price_log'] * df['num_ingredients']
    df['price_x_reviews'] = df['price_log'] * df['reviews_log']

    # 4. LIMIT SECONDARY CATEGORY TO TOP N
    top_secondary = df['secondary_category'].value_counts().nlargest(top_n_secondary).index
    df['secondary_category_top'] = df['secondary_category'].apply(
        lambda x: x if x in top_secondary else 'Other'
    )

    # 5. INGREDIENT FEATURE ENGINEERING
    ingredients_long = (
        df[['ingredients']]
        .copy()
        .assign(product_idx=lambda x: x.index)
        .assign(ingredient=lambda x: x['ingredients'].fillna('NA').astype(str).str.split(','))
        .explode('ingredient')
        .copy()
    )

    ingredients_long['ingredient'] = (
        ingredients_long['ingredient']
        .astype(str)
        .str.lower()
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.replace(r"^\[+", "", regex=True)
        .str.replace(r"^'+", "", regex=True)
        .str.replace(r"^\[\s*\+/-\s*", "", regex=True)
        .str.replace(r"^\+/-\s*", "", regex=True)
        .str.replace(r"\.+$", "", regex=True)
        .str.replace(r"\)+$", "", regex=True)
        .str.replace(r"\(.*", "", regex=True)
        .str.replace(r"\s*[\\/]\s*", "/", regex=True)
        .str.strip()
    )

    ingredients_long = ingredients_long[
        ~ingredients_long['ingredient'].isin(['', 'na', '1', 'none', 'nan', '+/-'])
    ].copy()

    ingredient_map = {
        'aqua': 'water',
        'eau': 'water',
        'water': 'water',
        'water/aqua': 'water',
        'aqua/water': 'water',
        'aqua/water/eau': 'water',
        'water/aqua/eau': 'water',
        'water//aqua//eau': 'water',
        'water\\aqua\\eau': 'water',
        'tocopherol': 'vitamin e',
        'tocopheryl acetate': 'vitamin e',
        'sodium hyaluronate': 'hyaluronic acid',
        'ci 77891': 'titanium dioxide',
        'titanium dioxide': 'titanium dioxide',
        'titanium dioxide/ci 77891': 'titanium dioxide',
        'ci 77891/titanium dioxide': 'titanium dioxide',
        'ci 77491': 'iron oxides',
        'ci 77492': 'iron oxides',
        'ci 77499': 'iron oxides',
        'iron oxides': 'iron oxides',
        'iron oxides/ci 77491': 'iron oxides',
        'iron oxides/ci 77492': 'iron oxides',
        'iron oxides/ci 77499': 'iron oxides',
        'parfum': 'fragrance',
        'perfume': 'fragrance',
        'fragrance/parfum': 'fragrance',
        'ci 15850': 'red 7 lake',
        'ci 42090': 'blue 1 lake',
        'simmondsia chinensis seed oil': 'jojoba oil',
        'helianthus annuus seed oil': 'sunflower oil',
        'zea mays starch': 'corn starch',
        'butyrospermum parkii butter': 'shea butter',
        'mica': 'mica',
        'talc': 'talc',
        'glycerin': 'glycerin',
        'butylene glycol': 'butylene glycol',
        'caprylyl glycol': 'caprylyl glycol',
        'phenoxyethanol': 'phenoxyethanol',
        'dimethicone': 'dimethicone'
    }

    ingredients_long['ingredient_clean'] = (
        ingredients_long['ingredient']
        .replace(ingredient_map)
        .astype(str)
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.replace(r"^\s+|\s+$", "", regex=True)
        .str.replace(r"^\[+", "", regex=True)
        .str.replace(r"^'+", "", regex=True)
        .str.replace(r"\.+$", "", regex=True)
        .str.replace(r"\)+$", "", regex=True)
        .str.strip()
    )

    ingredients_long = ingredients_long[
        ~ingredients_long['ingredient_clean'].isin(['', 'na', '1', 'none', 'nan', '+/-'])
    ].copy()

    ingredients_long = ingredients_long.drop_duplicates(
        subset=['product_idx', 'ingredient_clean']
    ).copy()

    # 6. SPLIT FIRST
    base_cols = [
        'reviews_log',
        'limited_edition',
        'online_only',
        'sephora_exclusive',
        'price_log',
        'num_ingredients',
        'price_x_num_ing',
        'price_x_reviews',
        'secondary_category_top'
    ]

    valid_rating_idx = df[df['rating'].notna()].index

    train_idx, temp_idx = train_test_split(valid_rating_idx, test_size=0.10, random_state=42)
    val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, random_state=42)

    df_train = df.loc[train_idx].copy()
    df_val = df.loc[val_idx].copy()
    df_test = df.loc[test_idx].copy()

    # 7. TOP INGREDIENTS FROM TRAIN ONLY
    top_50_ingredients = (
        ingredients_long.loc[ingredients_long['product_idx'].isin(train_idx), 'ingredient_clean']
        .value_counts()
        .head(50)
        .index
    )

    ingredients_long_top50 = ingredients_long[
        ingredients_long['ingredient_clean'].isin(top_50_ingredients)
    ].copy()

    def build_ingredient_text(df_subset):
        subset_long = (
            df_subset[['ingredients']]
            .copy()
            .assign(product_idx=lambda x: x.index)
            .assign(ingredient=lambda x: x['ingredients'].fillna('NA').astype(str).str.split(','))
            .explode('ingredient')
        )

        subset_long['ingredient'] = (
            subset_long['ingredient']
            .astype(str)
            .str.lower()
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .str.replace(r"^\[+", "", regex=True)
            .str.replace(r"^'+", "", regex=True)
            .str.replace(r"^\[\s*\+/-\s*", "", regex=True)
            .str.replace(r"^\+/-\s*", "", regex=True)
            .str.replace(r"\.+$", "", regex=True)
            .str.replace(r"\)+$", "", regex=True)
            .str.replace(r"\(.*", "", regex=True)
            .str.replace(r"\s*[\\/]\s*", "/", regex=True)
            .str.strip()
        )

        subset_long['ingredient_clean'] = (
            subset_long['ingredient']
            .replace(ingredient_map)
            .astype(str)
            .str.lower()
            .str.replace(r"\s+", " ", regex=True)
            .str.replace(r"^\s+|\s+$", "", regex=True)
            .str.replace(r"^\[+", "", regex=True)
            .str.replace(r"^'+", "", regex=True)
            .str.replace(r"\.+$", "", regex=True)
            .str.replace(r"\)+$", "", regex=True)
            .str.strip()
        )

        subset_long = subset_long[
            subset_long['ingredient_clean'].isin(top_50_ingredients)
        ].drop_duplicates(subset=['product_idx', 'ingredient_clean'])

        text = (
            subset_long.groupby('product_idx')['ingredient_clean']
            .apply(lambda x: ','.join(x))
        )

        return text.reindex(df_subset.index).fillna('')

    df_train['ingredients_clean'] = build_ingredient_text(df_train)
    df_val['ingredients_clean'] = build_ingredient_text(df_val)
    df_test['ingredients_clean'] = build_ingredient_text(df_test)

    # 8. EXPLICIT INGREDIENT FLAGS
    for subset in [df_train, df_val, df_test]:
        subset['has_fragrance'] = subset['ingredients_clean'].str.contains('fragrance', regex=False).astype(int)
        subset['has_hyaluronic_acid'] = subset['ingredients_clean'].str.contains('hyaluronic acid', regex=False).astype(int)

    # NEW: extra top ingredient flags
    existing_manual = {'fragrance', 'hyaluronic acid'}

    weak_ingredients = {
        'iron oxides',
        'titanium dioxide',
        'water',
        'vitamin e',
        'silica',
        'mica'
    }

    borderline_ingredients = {
        'phenoxyethanol',
        'caprylyl glycol',
        'butylene glycol'
    }

    top_flag_ingredients = [
        ing for ing in list(top_50_ingredients[:30])
        if ing not in existing_manual
        and ing not in weak_ingredients
        and ing not in borderline_ingredients
    ][:10]

    def sanitize_feature_name(text):
        return (
            str(text)
            .lower()
            .replace(' ', '_')
            .replace('/', '_')
            .replace('-', '_')
            .replace('.', '')
            .replace('(', '')
            .replace(')', '')
        )

    top_flag_feature_names = {
        ing: f"has_{sanitize_feature_name(ing)}"
        for ing in top_flag_ingredients
    }

    def add_top_ingredient_flags(df_subset):
        for ing in top_flag_ingredients:
            feature_name = top_flag_feature_names[ing]
            df_subset[feature_name] = df_subset['ingredients_clean'].str.contains(ing, regex=False).astype(int)
        return df_subset

    df_train = add_top_ingredient_flags(df_train)
    df_val = add_top_ingredient_flags(df_val)
    df_test = add_top_ingredient_flags(df_test)

    # 9. TF-IDF + SVD FIT ON TRAIN ONLY
    vectorizer = TfidfVectorizer(
        token_pattern=r'[^,]+',
        max_features=50,
        min_df=2,
        max_df=0.95
    )

    X_train_ing = vectorizer.fit_transform(df_train['ingredients_clean'])
    X_val_ing = vectorizer.transform(df_val['ingredients_clean'])
    X_test_ing = vectorizer.transform(df_test['ingredients_clean'])

    max_possible_components = min(
        n_svd_components,
        max(1, X_train_ing.shape[0] - 1),
        max(1, X_train_ing.shape[1] - 1)
    )

    svd = TruncatedSVD(n_components=max_possible_components, random_state=42)
    X_train_ing_svd = svd.fit_transform(X_train_ing)
    X_val_ing_svd = svd.transform(X_val_ing)
    X_test_ing_svd = svd.transform(X_test_ing)

    ingredients_svd_train = pd.DataFrame(
        X_train_ing_svd,
        columns=[f"ing_svd_{i+1}" for i in range(max_possible_components)],
        index=df_train.index
    )

    ingredients_svd_val = pd.DataFrame(
        X_val_ing_svd,
        columns=[f"ing_svd_{i+1}" for i in range(max_possible_components)],
        index=df_val.index
    )

    ingredients_svd_test = pd.DataFrame(
        X_test_ing_svd,
        columns=[f"ing_svd_{i+1}" for i in range(max_possible_components)],
        index=df_test.index
    )

    for i in range(max_possible_components + 1, n_svd_components + 1):
        ingredients_svd_train[f"ing_svd_{i}"] = 0.0
        ingredients_svd_val[f"ing_svd_{i}"] = 0.0
        ingredients_svd_test[f"ing_svd_{i}"] = 0.0

    ingredients_svd_train = ingredients_svd_train[[f"ing_svd_{i+1}" for i in range(n_svd_components)]]
    ingredients_svd_val = ingredients_svd_val[[f"ing_svd_{i+1}" for i in range(n_svd_components)]]
    ingredients_svd_test = ingredients_svd_test[[f"ing_svd_{i+1}" for i in range(n_svd_components)]]

    explained_var = svd.explained_variance_ratio_.sum()
    print(f"Ingredient SVD explained variance: {explained_var:.4f}")

    # 10. ENCODING + SCALING FIT ON TRAIN ONLY
    top_flag_cols = [top_flag_feature_names[ing] for ing in top_flag_ingredients]

    X_train_tab = pd.get_dummies(
        df_train[base_cols + ['has_fragrance', 'has_hyaluronic_acid'] + top_flag_cols],
        columns=['secondary_category_top'],
        drop_first=True,
        dtype=int
    )
    X_val_tab = pd.get_dummies(
        df_val[base_cols + ['has_fragrance', 'has_hyaluronic_acid'] + top_flag_cols],
        columns=['secondary_category_top'],
        drop_first=True,
        dtype=int
    )
    X_test_tab = pd.get_dummies(
        df_test[base_cols + ['has_fragrance', 'has_hyaluronic_acid'] + top_flag_cols],
        columns=['secondary_category_top'],
        drop_first=True,
        dtype=int
    )

    X_val_tab = X_val_tab.reindex(columns=X_train_tab.columns, fill_value=0)
    X_test_tab = X_test_tab.reindex(columns=X_train_tab.columns, fill_value=0)

    cols_to_scale = [
        'price_log',
        'reviews_log',
        'num_ingredients',
        'price_x_num_ing',
        'price_x_reviews'
    ]
    scaler = StandardScaler()

    X_train_tab[cols_to_scale] = scaler.fit_transform(X_train_tab[cols_to_scale])
    X_val_tab[cols_to_scale] = scaler.transform(X_val_tab[cols_to_scale])
    X_test_tab[cols_to_scale] = scaler.transform(X_test_tab[cols_to_scale])

    # 11. FINAL MATRICES
    X_train = pd.concat([X_train_tab, ingredients_svd_train], axis=1)
    X_val = pd.concat([X_val_tab, ingredients_svd_val], axis=1)
    X_test = pd.concat([X_test_tab, ingredients_svd_test], axis=1)

    X_train = X_train.loc[:, ~X_train.columns.duplicated()].copy()
    X_val = X_val.loc[:, ~X_val.columns.duplicated()].copy()
    X_test = X_test.loc[:, ~X_test.columns.duplicated()].copy()

    y_train_r = df.loc[train_idx, 'rating']
    y_val_r = df.loc[val_idx, 'rating']
    y_test_r = df.loc[test_idx, 'rating']

    y_train_l = df.loc[train_idx, 'log_loves']
    y_val_l = df.loc[val_idx, 'log_loves']
    y_test_l = df.loc[test_idx, 'log_loves']

    print(f"Pipeline Complete. Training features: {X_train.shape[1]}")
    print(f"Rating Training Samples: {len(X_train)}")
    print(f"Love Training Samples: {len(X_train)}")
    print("Final feature columns:")
    print(X_train.columns.tolist())

    return (
        X_train, X_val, X_test,
        y_train_r, y_val_r, y_test_r,
        y_train_l, y_val_l, y_test_l,
        ingredients_long, ingredients_long_top50,
        vectorizer, svd
    )

(
    X_train_r, X_val_r, X_test_r,
    y_train_r, y_val_r, y_test_r,
    y_train_l, y_val_l, y_test_l,
    ingredients_long, ingredients_long_top50,
    vectorizer, svd
) = run_product_pipeline(df_raw, top_n_secondary=5, n_svd_components=10)

Ingredient SVD explained variance: 0.4326
Pipeline Complete. Training features: 35
Rating Training Samples: 2095
Love Training Samples: 2095
Final feature columns:
['reviews_log', 'limited_edition', 'online_only', 'sephora_exclusive', 'price_log', 'num_ingredients', 'price_x_num_ing', 'price_x_reviews', 'has_fragrance', 'has_hyaluronic_acid', 'has_dimethicone', 'has_glycerin', 'has_caprylic_capric_triglyceride', 'has_synthetic_fluorphlogopite', 'has_tin_oxide', 'has_ethylhexylglycerin', 'has_blue_1_lake', 'has_polyethylene', 'has_yellow_5_lake', 'has_pentaerythrityl_tetra_di_t_butyl_hydroxyhydrocinnamate', 'secondary_category_top_Cheek', 'secondary_category_top_Eye', 'secondary_category_top_Face', 'secondary_category_top_Lip', 'secondary_category_top_Other', 'ing_svd_1', 'ing_svd_2', 'ing_svd_3', 'ing_svd_4', 'ing_svd_5', 'ing_svd_6', 'ing_svd_7', 'ing_svd_8', 'ing_svd_9', 'ing_svd_10']


## Save The Final Splits into Model folder

In [11]:
# 1. Define your S3 bucket and folder path
bucket_name = 'ads508-team1-sephora'
s3_model_path = 'Model/final_splits'

# 2. Local directory to temporarily store files before upload
local_dir = './temp_splits'
os.makedirs(local_dir, exist_ok=True)

# 3. Create a helper dictionary for all your datasets
# Note: Ensure these variables (X_train_r, etc.) exist in your memory
data_to_save = {
    "X_train_rating": pd.DataFrame(X_train_r), 
    "X_val_rating": pd.DataFrame(X_val_r), 
    "X_test_rating": pd.DataFrame(X_test_r),
    "y_train_rating": pd.DataFrame(y_train_r), 
    "y_val_rating": pd.DataFrame(y_val_r), 
    "y_test_rating": pd.DataFrame(y_test_r),
    
    "X_train_love": pd.DataFrame(X_train_r), 
    "X_val_love": pd.DataFrame(X_val_r), 
    "X_test_love": pd.DataFrame(X_test_r),
    "y_train_love": pd.DataFrame(y_train_l), 
    "y_val_love": pd.DataFrame(y_val_l), 
    "y_test_love": pd.DataFrame(y_test_l)
}

# 4. Save locally then upload to S3
s3_client = boto3.client('s3')

for name, df in data_to_save.items():
    local_file = f"{local_dir}/{name}.parquet"
    
    df = df.loc[:, ~df.columns.duplicated()]
    
    # Force all sparse columns to dense
    for col in df.columns:
        if pd.api.types.is_sparse(df[col]):
            df[col] = df[col].sparse.to_dense()
    
    df.to_parquet(local_file, index=False)
    
    s3_key = f"{s3_model_path}/{name}.parquet"
    s3_client.upload_file(local_file, bucket_name, s3_key)
    print(f"Successfully uploaded: s3://{bucket_name}/{s3_key}")

# 5. Clean up local files
shutil.rmtree(local_dir)

Successfully uploaded: s3://ads508-team1-sephora/Model/final_splits/X_train_rating.parquet
Successfully uploaded: s3://ads508-team1-sephora/Model/final_splits/X_val_rating.parquet
Successfully uploaded: s3://ads508-team1-sephora/Model/final_splits/X_test_rating.parquet
Successfully uploaded: s3://ads508-team1-sephora/Model/final_splits/y_train_rating.parquet
Successfully uploaded: s3://ads508-team1-sephora/Model/final_splits/y_val_rating.parquet
Successfully uploaded: s3://ads508-team1-sephora/Model/final_splits/y_test_rating.parquet
Successfully uploaded: s3://ads508-team1-sephora/Model/final_splits/X_train_love.parquet
Successfully uploaded: s3://ads508-team1-sephora/Model/final_splits/X_val_love.parquet
Successfully uploaded: s3://ads508-team1-sephora/Model/final_splits/X_test_love.parquet
Successfully uploaded: s3://ads508-team1-sephora/Model/final_splits/y_train_love.parquet
Successfully uploaded: s3://ads508-team1-sephora/Model/final_splits/y_val_love.parquet
Successfully uploade